# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\nPublished: {metadata.datePublished}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their `@id`s
record_sets = metadata.recordSet or []
print("Available record sets and their @id:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'unknown')}")

# For each record set, list the fields and their @id
for rs in record_sets:
    print(f"\nFields for record set @id: {rs['@id']}, name: {rs.get('name', 'unknown')}")
    if 'field' in rs:
        fields = rs['field']
        # fields can be a list of dicts (if >1 field) or a dict (if single field)
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            field_id = f.get('@id')
            name = f.get('name', 'unknown')
            dtype = f.get('dataType', 'unknown')
            print(f"  - @id: {field_id}, name: {name}, type: {dtype}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, select the main record set (by convention first/only one)
# Adapt this value for your dataset
record_sets = metadata.recordSet or []
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for record set @id: {record_set_id}")

if dataframes:
    main_record_set_id = record_set_ids[0]
    print(f"\nFields (@id) in main record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No dataframes extracted. Check record set configuration.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Replace with the actual field @id for a numeric field, e.g., coverage or "cr:coverage"
# Print available columns for guidance
print("Columns in the main DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())

# Try to select a common numeric field, e.g. coverage, MW, peptide_count
# Set the @id for the coverage field as example
# Use the actual field @id (e.g. 'cr:coverage', 'cr:MW', etc.)
numeric_field = None
possible_numeric_fields = ['cr:coverage', 'cr:MW', 'cr:peptide_count', 'cr:peptides', 'cr:Score', 'cr:Abundance']
# Find one present in the DataFrame
for f in possible_numeric_fields:
    if f in dataframes[main_record_set_id].columns:
        numeric_field = f
        break
if not numeric_field:
    raise ValueError("No numeric field found in DataFrame for demonstration.")
else:
    print(f"Using numeric field: {numeric_field}")

# Remove NaN and ensure numeric conversion
df = dataframes[main_record_set_id].copy()
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = df[numeric_field].quantile(0.75)  # Use upper quartile as demo threshold

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df[[numeric_field]].head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

# Grouping: try to use e.g. accession, description or other categorical field
group_field = None
possible_group_fields = ['cr:accession', 'cr:description', 'cr:modification']
for f in possible_group_fields:
    if f in df.columns:
        group_field = f
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field} showing mean {numeric_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# If another numeric field is available, do a scatter plot
other_numeric_field = None
for f in possible_numeric_fields:
    if f != numeric_field and f in df.columns:
        other_numeric_field = f
        break
if other_numeric_field:
    df[other_numeric_field] = pd.to_numeric(df[other_numeric_field], errors='coerce')
    plt.figure(figsize=(7,5))
    sns.scatterplot(x=numeric_field, y=other_numeric_field, data=df)
    plt.title(f"{numeric_field} vs {other_numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel(other_numeric_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides a detailed account of detected proteins from extracellular vesicles with fields for accession, description, coverage, peptides, molecular weight, and more (referenced by their `@id`).
- The top 25% (by coverage or similar numeric field) of proteins can be extracted and normalized for detailed comparison.
- Simple EDA and visualizations reveal the distribution and relationships between the quantitative fields.
- The rich Croissant schema enables reproducible and structured data access through unique `@id`s for every entity.

**Recommendation:** For further analysis, leverage domain knowledge to select relevant fields (`@id`s) and perform domain-specific statistical and visual exploration.